# Notebook 01 — Raw Data Ingestion & Source Validation

**Purpose:** Validate the 11 raw CSV files before loading them into MySQL.

**Checks:** file availability, encoding, delimiter, schema, parser readability, logical row count, and empty-file detection.

**Output:** a validated ingestion summary and a reference baseline containing the actual row and column counts used for MySQL reconciliation.

**Out of scope:** cleaning, transformation, profiling, deduplication, and business analysis.

In [1]:
# Import required libraries.

from pathlib import Path
from datetime import datetime
import csv
import logging
import time

import pandas as pd

In [2]:
# Initialize project paths, execution timing, and notebook logging.

NOTEBOOK_START_TIME = datetime.now()
EXECUTION_START = time.perf_counter()

PROJECT_DIR = Path.cwd().parent
RAW_DATA_DIR = PROJECT_DIR / "Raw_Data"
REFERENCE_DATA_DIR = PROJECT_DIR / "Reference_Data"
LOG_DIR = PROJECT_DIR / "Logs"

REFERENCE_DATA_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

LOG_FILE = LOG_DIR / "01_Raw_Data_Ingestion.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
        logging.StreamHandler(),
    ],
    force=True,
)

if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(
        f"Raw_Data folder was not found: {RAW_DATA_DIR}"
    )

print("=" * 80)
print("RAW DATA INGESTION & SOURCE VALIDATION")
print("=" * 80)
print(f"Started   : {NOTEBOOK_START_TIME:%Y-%m-%d %H:%M:%S}")
print(f"Project   : {PROJECT_DIR}")
print(f"Raw Data  : {RAW_DATA_DIR}")
print(f"Reference : {REFERENCE_DATA_DIR}")
print(f"Log File  : {LOG_FILE}")

logging.info("Raw data ingestion validation started.")

2026-10-13 12:10:54,014 | INFO | Raw data ingestion validation started.


RAW DATA INGESTION & SOURCE VALIDATION
Started   : 2026-10-13 12:10:54
Project   : C:\PORTFOLIO\olist_ecommerce_customer_acquisition_revenue_analytics
Raw Data  : C:\PORTFOLIO\olist_ecommerce_customer_acquisition_revenue_analytics\Raw_Data
Reference : C:\PORTFOLIO\olist_ecommerce_customer_acquisition_revenue_analytics\Reference_Data
Log File  : C:\PORTFOLIO\olist_ecommerce_customer_acquisition_revenue_analytics\Logs\01_Raw_Data_Ingestion.log


In [3]:
# Configure notebook display settings.

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [4]:
# Define the expected source files and source-faithful schemas.

DATASET_CONFIG = {
    # E-Commerce
    "olist_customers_dataset.csv": {
        "source": "E-Commerce",
        "table": "raw_customers",
        "columns": [
            "customer_id",
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state",
        ],
    },

    "olist_geolocation_dataset.csv": {
        "source": "E-Commerce",
        "table": "raw_geolocation",
        "columns": [
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng",
            "geolocation_city",
            "geolocation_state",
        ],
    },

    "olist_orders_dataset.csv": {
        "source": "E-Commerce",
        "table": "raw_orders",
        "columns": [
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
        ],
    },

    "olist_order_items_dataset.csv": {
        "source": "E-Commerce",
        "table": "raw_order_items",
        "columns": [
            "order_id",
            "order_item_id",
            "product_id",
            "seller_id",
            "shipping_limit_date",
            "price",
            "freight_value",
        ],
    },

    "olist_order_payments_dataset.csv": {
        "source": "E-Commerce",
        "table": "raw_order_payments",
        "columns": [
            "order_id",
            "payment_sequential",
            "payment_type",
            "payment_installments",
            "payment_value",
        ],
    },

    "olist_order_reviews_dataset.csv": {
        "source": "E-Commerce",
        "table": "raw_order_reviews",
        "columns": [
            "review_id",
            "order_id",
            "review_score",
            "review_comment_title",
            "review_comment_message",
            "review_creation_date",
            "review_answer_timestamp",
        ],
    },

    "olist_products_dataset.csv": {
        "source": "E-Commerce",
        "table": "raw_products",
        "columns": [
            "product_id",
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm",
        ],
    },

    "olist_sellers_dataset.csv": {
        "source": "E-Commerce",
        "table": "raw_sellers",
        "columns": [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state",
        ],
    },

    "product_category_name_translation.csv": {
        "source": "E-Commerce",
        "table": "raw_product_category_translation",
        "columns": [
            "product_category_name",
            "product_category_name_english",
        ],
    },

    # Marketing Funnel
    "olist_marketing_qualified_leads_dataset.csv": {
        "source": "Marketing Funnel",
        "table": "raw_marketing_qualified_leads",
        "columns": [
            "mql_id",
            "first_contact_date",
            "landing_page_id",
            "origin",
        ],
    },

    "olist_closed_deals_dataset.csv": {
        "source": "Marketing Funnel",
        "table": "raw_closed_deals",
        "columns": [
            "mql_id",
            "seller_id",
            "sdr_id",
            "sr_id",
            "won_date",
            "business_segment",
            "lead_type",
            "lead_behaviour_profile",
            "has_company",
            "has_gtin",
            "average_stock",
            "business_type",
            "declared_product_catalog_size",
            "declared_monthly_revenue",
        ],
    },
}

print(f"Source datasets configured: {len(DATASET_CONFIG)}")

logging.info(
    "Configured %s source datasets.",
    len(DATASET_CONFIG),
)

Source datasets configured: 11


2026-10-13 12:10:54,055 | INFO | Configured 11 source datasets.


In [5]:
# Define the expected source row counts for this dataset version.

EXPECTED_ROW_COUNTS = {
    "olist_customers_dataset.csv": 99441,
    "olist_geolocation_dataset.csv": 1000163,
    "olist_orders_dataset.csv": 99441,
    "olist_order_items_dataset.csv": 112650,
    "olist_order_payments_dataset.csv": 103886,
    "olist_order_reviews_dataset.csv": 99224,
    "olist_products_dataset.csv": 32951,
    "olist_sellers_dataset.csv": 3095,
    "product_category_name_translation.csv": 71,
    "olist_marketing_qualified_leads_dataset.csv": 8000,
    "olist_closed_deals_dataset.csv": 842,
}

if set(EXPECTED_ROW_COUNTS) != set(DATASET_CONFIG):
    raise ValueError(
        "Expected row-count baseline does not match the configured datasets."
    )

print(f"Expected row counts configured: {len(EXPECTED_ROW_COUNTS)}")

Expected row counts configured: 11


In [6]:
# Define source-file validation settings.

EXPECTED_ENCODING = "utf-8-sig"
EXPECTED_DELIMITER = ","
CHUNK_SIZE = 50_000

print(f"Expected encoding : {EXPECTED_ENCODING}")
print(f"Expected delimiter: {repr(EXPECTED_DELIMITER)}")
print(f"Chunk size        : {CHUNK_SIZE:,}")

Expected encoding : utf-8-sig
Expected delimiter: ','
Chunk size        : 50,000


In [7]:
# Define small functions used by the source validation process.

def validate_encoding(path):
    """Return True when the file can be read using the expected encoding."""
    
    try:
        with path.open(
            "r",
            encoding=EXPECTED_ENCODING,
            errors="strict",
            newline=""
        ) as file:
            while file.read(1_000_000):
                pass
        
        return True

    except UnicodeDecodeError:
        return False


def detect_delimiter(path):
    """Detect the CSV delimiter from the source file."""

    with path.open(
        "r",
        encoding=EXPECTED_ENCODING,
        errors="replace",
        newline=""
    ) as file:
        sample = file.read(100_000)

    try:
        return csv.Sniffer().sniff(
            sample,
            delimiters=",;\t|"
        ).delimiter

    except csv.Error:
        return None


def detect_line_ending(path):
    """Identify the dominant line-ending pattern for observation."""

    with path.open("rb") as file:
        sample = file.read(1_000_000)

    crlf = sample.count(b"\r\n")
    lf = sample.count(b"\n") - crlf
    cr = sample.count(b"\r") - crlf

    if crlf and not lf and not cr:
        return "CRLF"

    if lf and not crlf and not cr:
        return "LF"

    if cr and not crlf and not lf:
        return "CR"

    if crlf or lf or cr:
        return "Mixed"

    return "None"


def count_rows(path):
    """Count logical CSV data rows using chunked reading."""

    row_count = 0

    for chunk in pd.read_csv(
        path,
        sep=EXPECTED_DELIMITER,
        encoding=EXPECTED_ENCODING,
        chunksize=CHUNK_SIZE,
        low_memory=False,
        on_bad_lines="error",
    ):
        row_count += len(chunk)

    return row_count

In [8]:
# Validate all configured source files.

results = []

logging.info("Source validation started.")

for file_name, config in DATASET_CONFIG.items():

    path = RAW_DATA_DIR / file_name
    expected_rows = EXPECTED_ROW_COUNTS[file_name]

    result = {
        "Source": config["source"],
        "Table": config["table"],
        "File": file_name,
        "Exists": path.is_file(),
        "Size_MB": None,
        "Encoding_Valid": False,
        "Delimiter": None,
        "Delimiter_Match": False,
        "Line_Ending": None,
        "Rows": None,
        "Expected_Rows": expected_rows,
        "Row_Count_Match": False,
        "Columns": None,
        "Expected_Columns": len(config["columns"]),
        "Missing": "",
        "Unexpected": "",
        "Schema_Match": False,
        "Duplicate_Headers": None,
        "CSV_Readable": False,
        "Parser_Error": None,
        "Status": "FAIL",
    }

    # Check file availability.
    if not path.is_file():
        logging.error("Source file not found: %s", file_name)
        results.append(result)
        continue

    # Check file size.
    result["Size_MB"] = round(
        path.stat().st_size / 1024**2,
        2
    )

    if path.stat().st_size == 0:
        result["Parser_Error"] = "File is empty."
        logging.error("Empty source file: %s", file_name)
        results.append(result)
        continue

    try:
        # Validate encoding.
        result["Encoding_Valid"] = validate_encoding(path)

        # Detect and validate delimiter.
        result["Delimiter"] = detect_delimiter(path)
        result["Delimiter_Match"] = (
            result["Delimiter"] == EXPECTED_DELIMITER
        )

        # Record line ending for source metadata.
        result["Line_Ending"] = detect_line_ending(path)

        # Read header only.
        header = pd.read_csv(
            path,
            sep=EXPECTED_DELIMITER,
            encoding=EXPECTED_ENCODING,
            nrows=0,
        )

        actual_columns = header.columns.tolist()
        expected_columns = config["columns"]

        result["CSV_Readable"] = True
        result["Columns"] = len(actual_columns)

        result["Duplicate_Headers"] = int(
            header.columns.duplicated().sum()
        )

        result["Missing"] = ", ".join(
            column
            for column in expected_columns
            if column not in actual_columns
        )

        result["Unexpected"] = ", ".join(
            column
            for column in actual_columns
            if column not in expected_columns
        )

        # Exact comparison checks names and column order.
        result["Schema_Match"] = (
            actual_columns == expected_columns
        )

        # Count logical data rows.
        result["Rows"] = count_rows(path)

        result["Row_Count_Match"] = (
            result["Rows"] == expected_rows
        )

        # Final source readiness decision.
        checks = [
            result["Encoding_Valid"],
            result["Delimiter_Match"],
            result["CSV_Readable"],
            result["Schema_Match"],
            result["Duplicate_Headers"] == 0,
            result["Rows"] > 0,
            result["Row_Count_Match"],
        ]

        result["Status"] = (
            "PASS"
            if all(checks)
            else "FAIL"
        )

        logging.info(
            "%s | rows=%s | schema=%s | status=%s",
            file_name,
            result["Rows"],
            result["Schema_Match"],
            result["Status"],
        )

    except Exception as exc:

        result["Parser_Error"] = str(exc)
        result["Status"] = "FAIL"

        logging.exception(
            "Validation failed for %s",
            file_name
        )

    results.append(result)


ingestion_summary = pd.DataFrame(results)

print("=" * 80)
print("SOURCE VALIDATION RESULTS")
print("=" * 80)

display(ingestion_summary)

2026-10-13 12:10:54,146 | INFO | Source validation started.
2026-10-13 12:10:54,493 | INFO | olist_customers_dataset.csv | rows=99441 | schema=True | status=PASS
2026-10-13 12:10:56,163 | INFO | olist_geolocation_dataset.csv | rows=1000163 | schema=True | status=PASS
2026-10-13 12:10:56,726 | INFO | olist_orders_dataset.csv | rows=99441 | schema=True | status=PASS
2026-10-13 12:10:57,134 | INFO | olist_order_items_dataset.csv | rows=112650 | schema=True | status=PASS
2026-10-13 12:10:57,339 | INFO | olist_order_payments_dataset.csv | rows=103886 | schema=True | status=PASS
2026-10-13 12:10:58,094 | INFO | olist_order_reviews_dataset.csv | rows=99224 | schema=True | status=PASS
2026-10-13 12:10:58,233 | INFO | olist_products_dataset.csv | rows=32951 | schema=True | status=PASS
2026-10-13 12:10:58,283 | INFO | olist_sellers_dataset.csv | rows=3095 | schema=True | status=PASS
2026-10-13 12:10:58,314 | INFO | product_category_name_translation.csv | rows=71 | schema=True | status=PASS
2026-

SOURCE VALIDATION RESULTS


,Source,Table,File,Exists,Size_MB,Encoding_Valid,Delimiter,Delimiter_Match,Line_Ending,Rows,Expected_Rows,Row_Count_Match,Columns,Expected_Columns,Missing,Unexpected,Schema_Match,Duplicate_Headers,CSV_Readable,Parser_Error,Status
0,E-Commerce,raw_customers,olist_customers_dataset.csv,True,8.62,True,",",True,LF,99441,99441,True,5,5,,,True,0,True,None,PASS
1,E-Commerce,raw_geolocation,olist_geolocation_dataset.csv,True,58.44,True,",",True,LF,1000163,1000163,True,5,5,,,True,0,True,None,PASS
2,E-Commerce,raw_orders,olist_orders_dataset.csv,True,16.84,True,",",True,LF,99441,99441,True,8,8,,,True,0,True,None,PASS
3,E-Commerce,raw_order_items,olist_order_items_dataset.csv,True,14.72,True,",",True,LF,112650,112650,True,7,7,,,True,0,True,None,PASS
4,E-Commerce,raw_order_payments,olist_order_payments_dataset.csv,True,5.51,True,",",True,LF,103886,103886,True,5,5,,,True,0,True,None,PASS
5,E-Commerce,raw_order_reviews,olist_order_reviews_dataset.csv,True,13.78,True,",",True,CRLF,99224,99224,True,7,7,,,True,0,True,None,PASS
6,E-Commerce,raw_products,olist_products_dataset.csv,True,2.27,True,",",True,LF,32951,32951,True,9,9,,,True,0,True,None,PASS
7,E-Commerce,raw_sellers,olist_sellers_dataset.csv,True,0.17,True,",",True,LF,3095,3095,True,4,4,,,True,0,True,None,PASS
8,E-Commerce,raw_product_category_translation,product_category_name_translation.csv,True,0.00,True,",",True,CRLF,71,71,True,2,2,,,True,0,True,None,PASS
9,Marketing Funnel,raw_marketing_qualified_leads,olist_marketing_qualified_leads_dataset.csv,True,0.67,True,",",True,LF,8000,8000,True,4,4,,,True,0,True,None,PASS


In [9]:
# Summarize the source validation outcome.

summary = pd.DataFrame({
    "Metric": [
        "Source files checked",
        "Files passed",
        "Files failed",
        "Rows checked",
    ],
    "Value": [
        len(ingestion_summary),
        int((ingestion_summary["Status"] == "PASS").sum()),
        int((ingestion_summary["Status"] == "FAIL").sum()),
        int(ingestion_summary["Rows"].fillna(0).sum()),
    ],
})

display(summary)

,Metric,Value
0,Source files checked,11
1,Files passed,11
2,Files failed,0
3,Rows checked,1559764


In [10]:
# Display only source files requiring investigation.

failed_files_df = ingestion_summary[
    ingestion_summary["Status"] == "FAIL"
].copy()

print("=" * 80)
print("SOURCE VALIDATION ISSUES")
print("=" * 80)

if failed_files_df.empty:

    print("PASS — all source files passed validation.")

else:

    display(
        failed_files_df[
            [
                "Source",
                "Table",
                "File",
                "Rows",
                "Expected_Rows",
                "Columns",
                "Expected_Columns",
                "Delimiter",
                "Delimiter_Match",
                "Encoding_Valid",
                "Row_Count_Match",
                "Schema_Match",
                "Duplicate_Headers",
                "CSV_Readable",
                "Parser_Error",
                "Status",
            ]
        ]
    )

SOURCE VALIDATION ISSUES
PASS — all source files passed validation.


In [11]:
# Export the completed ingestion validation summary.

summary_path = (
    REFERENCE_DATA_DIR
    / "raw_data_ingestion_summary.csv"
)

ingestion_summary.to_csv(
    summary_path,
    index=False
)

print("=" * 80)
print("INGESTION SUMMARY EXPORTED")
print("=" * 80)
print(f"File: {summary_path}")

logging.info(
    "Ingestion summary exported: %s",
    summary_path
)

2026-10-13 12:10:58,537 | INFO | Ingestion summary exported: C:\PORTFOLIO\olist_ecommerce_customer_acquisition_revenue_analytics\Reference_Data\raw_data_ingestion_summary.csv


INGESTION SUMMARY EXPORTED
File: C:\PORTFOLIO\olist_ecommerce_customer_acquisition_revenue_analytics\Reference_Data\raw_data_ingestion_summary.csv


In [12]:
# Stop the workflow if any source file fails validation.

failed_files = ingestion_summary.loc[
    ingestion_summary["Status"] != "PASS",
    "File"
].tolist()

if failed_files:

    logging.error(
        "Raw ingestion validation failed: %s",
        failed_files
    )

    raise RuntimeError(
        "RAW INGESTION VALIDATION FAILED.\n\n"
        "Fix the following source files before "
        "continuing to MySQL raw-table loading:\n\n"
        + "\n".join(
            f"- {file_name}"
            for file_name in failed_files
        )
    )

print("=" * 80)
print("RAW INGESTION GATE: PASS")
print("=" * 80)
print(
    "All configured source files passed the required "
    "source validation checks."
)

logging.info(
    "Raw ingestion validation gate passed."
)

2026-10-13 12:10:58,554 | INFO | Raw ingestion validation gate passed.


RAW INGESTION GATE: PASS
All configured source files passed the required source validation checks.


In [13]:
# NOTEBOOK SUMMARY

EXECUTION_DURATION_SECONDS = (
    time.perf_counter() - EXECUTION_START
)

NOTEBOOK_END_TIME = datetime.now()

total_files = len(ingestion_summary)

passed_files = int(
    (ingestion_summary["Status"] == "PASS").sum()
)

failed_count = int(
    (ingestion_summary["Status"] == "FAIL").sum()
)

total_rows = int(
    ingestion_summary["Rows"]
    .fillna(0)
    .sum()
)

print("=" * 80)
print("NOTEBOOK EXECUTION SUMMARY")
print("=" * 80)

print(f"Source files checked : {total_files}")
print(f"Files passed         : {passed_files}")
print(f"Files failed         : {failed_count}")
print(f"Logical rows checked : {total_rows:,}")
print(f"Started              : {NOTEBOOK_START_TIME:%Y-%m-%d %H:%M:%S}")
print(f"Completed            : {NOTEBOOK_END_TIME:%Y-%m-%d %H:%M:%S}")
print(
    f"Execution duration   : "
    f"{EXECUTION_DURATION_SECONDS:,.2f} seconds"
)
print(f"Summary file         : {summary_path}")
print(f"Log file             : {LOG_FILE}")

logging.info(
    "Notebook completed | files=%s | passed=%s | failed=%s | "
    "rows=%s | duration_seconds=%.2f",
    total_files,
    passed_files,
    failed_count,
    total_rows,
    EXECUTION_DURATION_SECONDS,
)

2026-10-13 12:10:58,583 | INFO | Notebook completed | files=11 | passed=11 | failed=0 | rows=1559764 | duration_seconds=4.57


NOTEBOOK EXECUTION SUMMARY
Source files checked : 11
Files passed         : 11
Files failed         : 0
Logical rows checked : 1,559,764
Started              : 2026-10-13 12:10:54
Completed            : 2026-10-13 12:10:58
Execution duration   : 4.57 seconds
Summary file         : C:\PORTFOLIO\olist_ecommerce_customer_acquisition_revenue_analytics\Reference_Data\raw_data_ingestion_summary.csv
Log file             : C:\PORTFOLIO\olist_ecommerce_customer_acquisition_revenue_analytics\Logs\01_Raw_Data_Ingestion.log
